# 05 · Flights, climbs and glides

Analyses every unique competition track in the catalog (notebook 04) and adds to `database/`:

| Table | One row per |
|---|---|
| `flights` | track: takeoff, landing, duration, height, number of thermals, mean climb rate, share of time circling (or why no flight was found) |
| `climbs` | circling climb: where, when, base and top, gain, climb rate, turns and direction, whether it is a thermal |
| `glides` | stretch between climbs: distance, height lost, speed, glide ratio |
| `fixes/` | one file per task with each flight's fixes every 5 s (position, altitude, vertical speed, circling) |

Each track is cleaned first (GPS spikes and altitude glitches), and the flight is separated from the drive up, the wait at launch and the retrieve drive that live-tracking archives often include. The method and its thresholds are in `pta/flight.py`; the tables are described in `docs/database.md`.

About 100,000 tracks take one to two hours, mostly reading the packages from Drive. Each task is cached, so **if Colab disconnects, Run All again** and it carries on.

## 1. Settings

In [ ]:
DRIVE_FOLDER_ID = "1_JO0XMMQ4NKVfpeA8uKQRkZZPNC4sqZ7"   # project folder in Google Drive (the id at the end of its link)
WORKERS = 2          # processes analysing tracks (Colab usually has 2 CPUs)
REPO_URL = "https://github.com/RobertojonesFAI/Pre-task-analyzer.git"
REPO = "/content/Pre-task-analyzer"

## 2. Mount Drive and get the code

In [ ]:
import os, shutil, subprocess
from google.colab import auth, drive
from googleapiclient.discovery import build

drive.mount("/content/drive")
auth.authenticate_user()
service = build("drive", "v3")
FIELDS = "id, name, parents, driveId, ownedByMe"


def path_up(file_id, root):
    """Folder names from a file up to the top of its drive, joined under root."""
    names, current = [], file_id
    while True:
        meta = service.files().get(fileId=current, fields=FIELDS, supportsAllDrives=True).execute()
        if not meta.get("parents"):
            return os.path.join(root, *reversed(names))
        names.append(meta["name"])
        current = meta["parents"][0]


def drive_path(folder_id):
    """A writable local path for a Drive folder, from its id."""
    meta = service.files().get(fileId=folder_id, fields=FIELDS, supportsAllDrives=True).execute()
    if meta.get("driveId"):  # in a shared drive
        drive_name = service.drives().get(driveId=meta["driveId"]).execute()["name"]
        return path_up(folder_id, f"/content/drive/Shareddrives/{drive_name}")
    if meta.get("ownedByMe"):  # in my own Drive
        return path_up(folder_id, "/content/drive/MyDrive")
    # Shared with me: write through a shortcut to it in my Drive
    page = None
    while True:
        res = service.files().list(
            q="mimeType='application/vnd.google-apps.shortcut' and trashed=false",
            fields="nextPageToken, files(id, shortcutDetails)", pageToken=page).execute()
        for f in res.get("files", []):
            if f.get("shortcutDetails", {}).get("targetId") == folder_id:
                return path_up(f["id"], "/content/drive/MyDrive")
        page = res.get("nextPageToken")
        if not page:
            raise SystemExit(f"'{meta['name']}' is shared with you. Add a shortcut to it in your Drive "
                             "(right click -> Organize -> Add shortcut), then rerun this cell.")


ROOT = drive_path(DRIVE_FOLDER_ID)
DB = os.path.join(ROOT, "database")
assert os.path.isfile(os.path.join(DB, "tracks.parquet")), "Run notebook 04 first: database/tracks.parquet is missing"
print("Database in", DB)

if os.path.exists(REPO):
    subprocess.run(["git", "-C", REPO, "pull", "-q"], check=True)
else:
    subprocess.run(["git", "clone", "-q", REPO_URL, REPO], check=True)


def run(*cmd):
    proc = subprocess.Popen(["python3", "-u", *cmd], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in proc.stdout:
        print(line, end="")
    return proc.wait()

## 3. Analyse the tracks

Prints progress every 25 tasks and a summary at the end. Tasks already analysed are skipped, except those where a track failed with an error, which are tried again (`--retry-errors`). After a change to the flight analysis itself, add `"--redo"` to analyse every track again.

In [ ]:
run(f"{REPO}/scripts/build_flights.py", DB, "--root", ROOT, "--workers", str(WORKERS), "--retry-errors")

## 4. A first look

In [ ]:
import duckdb

db = duckdb.connect()
for name in ["tasks", "results", "tracks", "flights", "climbs", "glides"]:
    db.execute(f"CREATE VIEW {name} AS SELECT * FROM read_parquet('{DB}/{name}.parquet')")

# Flights per year and source, with typical duration and climb rate
db.sql("""
    SELECT substr(k.date, 1, 4) AS year, f.source, count(*) AS flights,
           round(median(f.duration_s) / 3600, 1) AS median_hours,
           round(median(f.mean_thermal_rate_ms), 2) AS median_climb_ms
    FROM flights f JOIN tasks k USING (task_key)
    WHERE f.error IS NULL
    GROUP BY ALL ORDER BY year, f.source
""").df()

In [ ]:
# Climb rate and flight time by glider class, for pilots with a linked result
db.sql("""
    SELECT coalesce(r.glider_cert, 'unknown') AS class, count(*) AS flights,
           round(median(f.mean_thermal_rate_ms), 2) AS median_climb_ms,
           round(median(f.circling_share), 2) AS median_circling,
           round(avg(CASE WHEN r.status = 'goal' THEN 1 ELSE 0 END), 2) AS goal_share
    FROM flights f JOIN results r USING (result_id)
    WHERE f.error IS NULL
    GROUP BY ALL ORDER BY flights DESC
""").df()

In [ ]:
# Where the climbs are: one task's thermals on a map (change TASK to any task_key)
import folium

TASK = db.sql("SELECT task_key FROM flights WHERE error IS NULL GROUP BY 1 ORDER BY count(*) DESC LIMIT 1").df().task_key[0]
th = db.sql(f"SELECT lat, lon, rate_ms FROM climbs WHERE task_key = '{TASK}' AND is_thermal").df()
m = folium.Map(location=[th.lat.mean(), th.lon.mean()], zoom_start=10, tiles="OpenTopoMap")
for r in th.itertuples():
    folium.CircleMarker([r.lat, r.lon], radius=2 + r.rate_ms, color="#c0392b", weight=0,
                        fill=True, fill_opacity=0.35).add_to(m)
print(TASK, len(th), "thermals")
m